In [ ]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError

from methods import *



# Define constants
year = 2023
file_type = 'AMY'
save_folder = f'epws_wmo_{year}'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', dtype={f'EPW_file_name_{year}': str, f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame starting from the specified index
for index, row in zipcodes.iloc[18470:].iterrows():
    # if float(row.get(f"distance_location_station_miles_{year}")) < 50:
    #     continue
    print(index)


    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # print(lat)
    # print(lon)
    # print(row['city'])

    # Retrieve data for the current location
    # retrieve_status, distance, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations, flags = run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    if retrieve_info_closest_other_locations:
        try:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            retrieve_status, hdd, cdd, flags= retrieve_info_other_location(wmo, zipcodes, year)
        except IndexError:
            # retrieve_status, distance, hdd, cdd = retrieve_info_other_location(wmo, zipcodes, year)
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)




    # Validate that retrieve_status is a boolean
    if not isinstance(bool(retrieve_status), bool):
        raise TypeError(f"retrieve_status is not a boolean. Actual value: {retrieve_status}. Program stopped.")
    
    [distance_mi, lat_station, lon_station] = retrieve_distance_station_location(wmo, lat, lon)

    # Update the DataFrame only if the cell is empty or contains a placeholder (like 'nan')
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", f"{wmo}_{year}.epw")
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)  # Convert from meters to miles
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)
    update_if_missing(zipcodes, index, f"Tdb_holes_{year}", flags[6])
    update_if_missing(zipcodes, index, f"Tdew_holes_{year}", flags[7])
    update_if_missing(zipcodes, index, f"RH_holes_{year}", flags[8])

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        # Save the DataFrame to the CSV file
        zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)

        # Reopen the file to ensure the latest version is loaded
        # zipcodes = pd.read_csv('resources/zip_code_list.csv', dtype={f'Do we have data for {year}?': str, f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)

# KVHN0
# KSUN
# 74611


In [1]:
import os, sys, math, ssl, io, pytz, numpy as np, pandas as pd, requests
from datetime import datetime, timedelta, date
from timezonefinder import TimezoneFinder
from meteostat import Stations, Hourly
from isd import Batch
from scp import SCPClient
import paramiko
import calendar
from pandas.errors import EmptyDataError

from methods import *

# Define constants
year = 2023
file_type = 'AMY'
save_folder = f'epws_wmo_{year}'

# Check if the 'zipcodes' variable is already defined
if 'zipcodes' not in globals():
    # Load the zip codes CSV only if 'zipcodes' is not already defined
    zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', 
                           dtype={f'EPW_file_name_{year}': str, 
                                  f'weather_station_wmo_{year}': str})

# Initialize a counter for iterations
counter = 0

# Process each row in the DataFrame where 'EnergyPlus Status' is 'Bad'
for index, row in zipcodes[zipcodes['EnergyPlus Status'] == 'Bad'].iterrows():
    
    print(index)

    zip_code = str(row['zip0']).zfill(5)  # Ensure the zip code is a string and pad with leading zeros if needed
    print(zip_code)
    lat = row['lat']
    lon = row['lng']
    save_name = None

    # Retrieve data for the current location
    retrieve_status, wmo, hdd, cdd, latitude_station, longitude_station, retrieve_info_closest_other_locations, flags = \
        run_individual_location(lat, lon, year, file_type, save_folder, save_name)
    
    if retrieve_info_closest_other_locations:
        try:
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)
        except IndexError:
            retrieve_status, hdd, cdd, flags = retrieve_info_other_location(wmo, zipcodes, year)

    # Validate that retrieve_status is a boolean
    if not isinstance(bool(retrieve_status), bool):
        raise TypeError(f"retrieve_status is not a boolean. Actual value: {retrieve_status}. Program stopped.")
    
    # Calculate distance between station and location
    distance_mi, lat_station, lon_station = retrieve_distance_station_location(wmo, lat, lon)

    # Update the DataFrame only if the cell is empty or NaN
    update_if_missing(zipcodes, index, f"EPW_file_name_{year}", retrieve_status)
    update_if_missing(zipcodes, index, f"distance_location_station_miles_{year}", distance_mi)
    update_if_missing(zipcodes, index, f"weather_station_wmo_{year}", wmo)
    update_if_missing(zipcodes, index, f"hdd_base65F_{year}", hdd)
    update_if_missing(zipcodes, index, f"cdd_base65F_{year}", cdd)
    update_if_missing(zipcodes, index, f"Tdb_holes_{year}", flags[6])
    update_if_missing(zipcodes, index, f"Tdew_holes_{year}", flags[7])
    update_if_missing(zipcodes, index, f"RH_holes_{year}", flags[8])

    # Increment the counter
    counter += 1

    # Every 10 iterations, save the DataFrame and reopen it
    if counter % 10 == 0:
        zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)
        # Reopen the file if needed
        # zipcodes = pd.read_csv(f'resources/zip_code_list_{year}.csv', 
        #                       dtype={f'EPW_file_name_{year}': str, 
        #                              f'weather_station_wmo_{year}': str})

# After the loop is done, ensure the latest state is saved
zipcodes.to_csv(f'resources/zip_code_list_{year}.csv', index=False)


Debug

In [3]:

def get_noaa_merra2_data(lat, lon, year, file_type, save_folder):
    """
    Retrieves NOAA and MERRA2 data for a specific location and year.
    """
    retrieve_status = True
    data_noaa, tz, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    # data_noaa, tz, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists, incomplete_timeseries = get_data_noaa(lat, lon, year, save_folder)
    if epw_exists:
        df_merged = ''
        retrieve_status = False
        # distance = ''
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = ''
        epw_exists = True
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    elif incomplete_timeseries:
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        wmo = ''
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    try:
        data_noaa_tz_adj = filter_dataframe_by_date(convert_utc_to_local(data_noaa, tz), datetime(year, 1, 1), datetime(year+1, 1, 1))
    except AttributeError:
        # print("We don't have NOAA data for this location/year")
        df_merged = ''
        retrieve_status = False
        # distance = np.nan
        hdd = ''
        cdd = ''
        latitude_station = ''
        longitude_station = ''
        info_dict = '' 
        epw_exists = False
        # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
        return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists

    info_dict = {
    'timeshift': get_time_shift(tz),
    'elevation': elevation,
    'wmo': wmo,
    'station_name': station_name,
    'state': state,
    'country': country,
    'lat': latitude_station,
    'lon': longitude_station,
    'weather_file_type': file_type
    }

    data_noaa_tz_adj_h = data_noaa_tz_adj.resample('H').mean()
    data_noaa_tz_adj_h_interpolated = data_noaa_tz_adj_h.interpolate(method='linear', limit=3, limit_direction='both')
    hdd, cdd = calculate_hdd_cdd(data_noaa_tz_adj_h_interpolated, 'temp')
    try:
        df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
    except EmptyDataError:
        try:
            df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
        except EmptyDataError:
            try:
                df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
            except EmptyDataError:
                try:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)
                except EmptyDataError:
                    df_merra2, header_merra2 = get_parameters_MERRA2(latitude_station, longitude_station, year)


    df_merged = merge_data(df_merra2, data_noaa_tz_adj_h_interpolated)

    # Check for empty cells in df_merged
    # if df_merged.isnull().any().any():
    #     raise ValueError("The merged DataFrame (df_merged) contains empty cells. Stopping execution.")


    # return df_merged, retrieve_status, info_dict, distance, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists
    return df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists


[df_merged, retrieve_status, info_dict, hdd, cdd, wmo, latitude_station, longitude_station, epw_exists] = get_noaa_merra2_data(37.73096, -115.24786, 2022, 'AMY', '')





In [5]:
df_merged.to_csv('TEST_.csv')

In [ ]:
print(zipcodes.columns)

In [ ]:
meteostat_df = pd.read_csv('resources/meteostat_stats.csv')
zipcodes = pd.read_csv('resources/zip_code_list.csv')
meteostat_df


In [ ]:
zip_row.columns

In [ ]:
import math

# Function to calculate the distance between two lat/lon points using the Haversine formula
def haversine(lat1, lon1, lat2, lon2):
    # Radius of the Earth in miles
    R = 3958.8
    
    # Convert degrees to radians
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    
    # Distance in miles
    return R * c


# Iterate over each row in zip_df
for idx, zip_row in zipcodes.iterrows():
    # print(idx)

    if bool(zip_row['Do we have data for 2022?']):
        wmo_code = zip_row['weather_station_wmo_2022']

        # print(wmo_code)
        
        # Look for a matching row in meteostat_df using WMO or ICAO code
        # station_row = meteostat_df[(meteostat_df['wmo'].astype(str) == wmo_code) | (meteostat_df['icao'].astype(str) == wmo_code)]
        try:
            station_row = meteostat_df[(meteostat_df['id'].astype(str) == str(wmo_code))]
        except ValueError:
            try:
                station_row = meteostat_df[(meteostat_df['id'].astype(str) == wmo_code[:4])]
            except TypeError:
                print('----------------------------')
                print(wmo_code)
                print(zip_row['zip0'])
                print(zip_row['Do we have data for 2022?'])
                continue
                # station_row = meteostat_df[(meteostat_df['icao'].astype(str) == wmo_code[:4])]
               
        



        if not station_row.empty:
            # Extract lat/lon from meteostat_df
            lat_stat = station_row['latitude'].values[0]
            lon_stat = station_row['longitude'].values[0]
            
            # Extract lat/lon from zip_df
            lat_zip = zip_row['lat']
            lon_zip = zip_row['lng']
            
            # Calculate the distance in miles
            distance = haversine(lat_zip, lon_zip, lat_stat, lon_stat)
            
            # Save the calculated distance in the new column
            zipcodes.at[idx, 'distance_location_station_miles_2022___'] = distance

# Show the updated zip_df with distances
zipcodes.head()


In [ ]:
meteostat_df[(meteostat_df['wmo'] == int('71345'))]

In [7]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)


In [19]:


Stations().nearby(48.72526, -111.36528).fetch().to_csv('resources/meteostat_stats.csv', index=True)

In [ ]:
def get_data_noaa(lat, lon, year, save_folder):
    """
    Fetches NOAA data for a given location and year, handling timezones and missing data.
    """
    # Disable SSL verification
    ssl._create_default_https_context = ssl._create_unverified_context

    start = datetime(year - 1, 12, 31)
    end = datetime(year + 1, 1, 2)

    stations = Stations().nearby(lat, lon)

    epw_exists = False
    station_number = 0
    len_data = 0

    incomplete_timeseries = True
    while incomplete_timeseries:
        station_number += 1
        wmo = fix_wmo(str(stations.fetch(station_number).index.values[-1]))
        # First check if EPW already exists
        if check_epw_exists(save_folder, year, wmo):
            epw_exists = True
            incomplete_timeseries = False
            break
        data = Hourly(stations.fetch(station_number), start, end, model=True).fetch()
        len_data = len(data)
        missing_hours_num, largest_consecutive_group = check_missing_hours(year, data)
        if (len_data > 8000) & (largest_consecutive_group <= 3):
            incomplete_timeseries = False
        # distance = stations.fetch(station_number)['distance'].values[-1]
        # # Let's stop after 100mi
        # if distance > 160000:
        #     break

    if epw_exists | incomplete_timeseries:
        data = ''
        timezone = ''
        # distance = ''
        elevation = ''
        station_name = ''
        state = ''
        country = ''
        latitude_station = ''
        longitude_station = ''
                
    else:
        station_info = stations.fetch(station_number)
        timezone = station_info['timezone'].values[-1]
        elevation = station_info['elevation'].values[-1]
        # distance = stations.fetch()['distance'].values[-1]
        wmo = fix_wmo(str(station_info.index.values[-1]))
        station_name = station_info['name'].values[-1]
        state = station_info['region'].values[-1]
        country = station_info['country'].values[-1]
        latitude_station = station_info['latitude'].values[-1]
        longitude_station = station_info['longitude'].values[-1]

    # return data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries
    return data, timezone, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries



lat = 42.06259
lon = -72.62589

data, timezone, distance, elevation, wmo, station_name, state, country, latitude_station, longitude_station, epw_exists,incomplete_timeseries = get_data_noaa(lat, lon, 2022, '')
data.head(5)

In [ ]:
data.interpolate(method='linear', limit=3, limit_direction='forward')

In [ ]:
zipcodes.head(20)

In [ ]:

file_type = 'AMY'
lat = 41.766595
lon = -88.318735
year = 2023
name= 'TEST_2023'
output_name = name + '.epw'

# Run your existing code with these parameters
data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
data_meteostat_merra2.to_csv(output_name, header=False, index=False)
with open(output_name, 'r') as original_file:
    data_content = original_file.read()
header_lines = create_header(data_meteostat_merra2, year, info_dict)
with open(output_name, 'w') as new_file:
    new_file.write("\n".join(header_lines) + "\n" + data_content)


In [ ]:
import sys
from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QLineEdit, QPushButton, QVBoxLayout, QGridLayout, QMessageBox
import pandas as pd
import numpy as np
import paramiko
from scp import SCPClient
from isd import Batch
from meteostat import Stations, Hourly
from timezonefinder import TimezoneFinder
from datetime import datetime, timedelta, date
import pytz
import requests
import ssl
import io

# Assuming all the previous functions are defined above or imported from another module


def run_individual_location(output_name, lat, lon, year, file_type, name):
    try:
        # Run your existing code with these parameters
        data_meteostat_merra2, missing_dates, info_dict = get_noaa_merra2_data(lat, lon, year, file_type)
        data_meteostat_merra2.to_csv(output_name, header=False, index=False)
        with open(output_name, 'r') as original_file:
            data_content = original_file.read()
        header_lines = create_header(data_meteostat_merra2, year, info_dict)
        with open(output_name, 'w') as new_file:
            new_file.write("\n".join(header_lines) + "\n" + data_content)
        QMessageBox.information(window, "Success", f"Data saved successfully to {output_name}")
    except Exception as e:
        QMessageBox.critical(window, "Error", f"An error occurred: {str(e)}")


def on_run_clicked():
    lat = float(lat_input.text())
    lon = float(lon_input.text())
    year = int(year_input.text())
    file_type = file_type_input.text()
    name = name_input.text()
    output_name = output_name_input.text()
    
    run_individual_location(output_name, lat, lon, year, file_type, name)


# Initialize the application
app = QApplication(sys.argv)

# Create the main window
window = QWidget()
window.setWindowTitle("NOAA MERRA2 Data Processor")
window.setGeometry(100, 100, 400, 300)

# Create a grid layout
layout = QGridLayout()

# Add widgets for input fields
layout.addWidget(QLabel("Latitude:"), 0, 0)
lat_input = QLineEdit()
layout.addWidget(lat_input, 0, 1)
lat_input.setText("41.766595")

layout.addWidget(QLabel("Longitude:"), 1, 0)
lon_input = QLineEdit()
layout.addWidget(lon_input, 1, 1)
lon_input.setText("-88.318735")

layout.addWidget(QLabel("Year:"), 2, 0)
year_input = QLineEdit()
layout.addWidget(year_input, 2, 1)
year_input.setText("2023")

layout.addWidget(QLabel("File Type:"), 3, 0)
file_type_input = QLineEdit()
layout.addWidget(file_type_input, 3, 1)
file_type_input.setText("AMY")

layout.addWidget(QLabel("Name:"), 4, 0)
name_input = QLineEdit()
layout.addWidget(name_input, 4, 1)
name_input.setText("TEST_2023")

layout.addWidget(QLabel("Output File Name:"), 5, 0)
output_name_input = QLineEdit()
layout.addWidget(output_name_input, 5, 1)
output_name_input.setText("TEST_2023.epw")

# Add a run button
run_button = QPushButton("Run")
run_button.clicked.connect(on_run_clicked)
layout.addWidget(run_button, 6, 0, 1, 2)

# Set the layout for the main window
window.setLayout(layout)

# Show the window
window.show()

# Run the application's main loop
sys.exit(app.exec_())


## Figure out Zip Codes

In [ ]:
import pandas as pd
import numpy as np

# Define a function to calculate the Haversine distance between two points in km
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula to calculate the distance
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

# 1) Open the file resources/zip_code_list.csv as a dataframe and call it zipcodes
zipcodes = pd.read_csv('resources/zip_codes_list.csv')

# 2) Open design_conditions.csv as a dataframe and call it dc
dc = pd.read_csv('resources/meteostat_stats.csv')

# Ensure latitude and longitude columns are in float format
zipcodes['lat'] = zipcodes['lat'].astype(float)
zipcodes['lng'] = zipcodes['lng'].astype(float)
dc['latitude'] = dc['latitude'].astype(float)
dc['longitude'] = dc['longitude'].astype(float)

# 3) Initialize columns in zipcodes dataframe for storing results
zipcodes['Distance'] = np.nan
zipcodes['Location'] = ""

# 4) Loop through all the rows in zipcodes
for idx, row in zipcodes.iterrows():
    lat1 = row['lat']
    lon1 = row['lng']

    # Calculate the distance to each location in the dc dataframe
    dc['Distance'] = dc.apply(lambda x: haversine(lat1, lon1, x['latitude'], x['longitude']), axis=1)

    # 5) Find the closest location in dc
    closest_location = dc.loc[dc['Distance'].idxmin()]

    # 6) Update the zipcodes dataframe with the closest location's details
    zipcodes.at[idx, 'Distance'] = closest_location['Distance']
    zipcodes.at[idx, 'Location'] = closest_location['name']

# Display the updated dataframe
zipcodes.to_csv('resources/updated_zip_code_list_again.csv', index=False)

In [ ]:
# Ensure the 'zip' column is a string and pad with zeros to make it 5 digits
zipcodes['zip0'] = zipcodes['zip'].astype(str).str.zfill(5)

zipcodes


In [ ]:
zipcodes.to_csv('resources/zip_code_list.csv', index=False)

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL of the dataset containing ZIP codes and their respective coordinates
url = "https://raw.githubusercontent.com/scpike/us-state-county-zip/master/geo-data.csv"

# Fetching the CSV file from the URL
response = requests.get(url)
response.raise_for_status()  # Raises an error for bad responses

# Reading the CSV data into a pandas DataFrame
data = pd.read_csv(StringIO(response.text))


data.to_csv('resources/zip_codes_list.csv')


In [ ]:
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile

# URL of a dataset containing ZIP codes, latitude, and longitude
url = "https://simplemaps.com/static/data/us-zips/1.74/basic/simplemaps_uszips_basicv1.74.zip"

# Download the ZIP file with SSL verification disabled
response = requests.get(url, verify=False)  # Bypass SSL certificate verification
response.raise_for_status()  # Check if the request was successful

# Unzip the file and read the CSV
with ZipFile(BytesIO(response.content)) as zip_file:
    # Extract the CSV file within the ZIP
    with zip_file.open('uszips.csv') as file:
        zip_code_data = pd.read_csv(file)

# # Display the DataFrame to verify the contents
# print(zip_code_data.head())

# # Save the DataFrame to a local CSV file
zip_code_data.to_csv('resources/zip_codes_list.csv', index=False)
# print("Data saved to 'us_zip_codes_with_coordinates.csv'.")


In [ ]:
zip_code_data